# GSM8K DAPO training on Colab

This notebook runs `RL_GRPO_train/configs/qwen25_3b_sft_dapo.yaml`. The YAML is a GRPO-controlled config plus the four DAPO components currently supported by TRL: token-level loss, overlong filtering, clip-higher, and soft overlong punishment. Dynamic sampling is a DAPO component in the paper, but TRL documentation marks it as unsupported.

In [1]:
!nvidia-smi

Tue May  5 02:03:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             56W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path
import os

candidates = [
    Path('/content/drive/MyDrive/HPML_project'),
    Path('/content/drive/MyDrive/HPML_project/project'),
    Path.cwd(),
]
PROJECT_DIR = next((p for p in candidates if (p / 'RL_GRPO_train').exists() and (p / 'RL_common').exists()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError('Cannot find project root. Mount Drive or edit PROJECT_DIR manually.')

os.chdir(PROJECT_DIR)
print('PROJECT_DIR =', PROJECT_DIR)
!pwd

PROJECT_DIR = /content/drive/MyDrive/HPML_project
/content/drive/MyDrive/HPML_project


## Tokens and SwanLab

In [4]:
import getpass
import os

USE_COLAB_SECRETS = True

if USE_COLAB_SECRETS:
    from google.colab import userdata
    for key in ['HF_TOKEN', 'SWANLAB_API_KEY']:
        value = userdata.get(key)
        if value:
            os.environ[key] = value
else:
    for key in ['HF_TOKEN', 'SWANLAB_API_KEY']:
        if not os.environ.get(key):
            value = getpass.getpass(f'{key} (press Enter to skip): ')
            if value:
                os.environ[key] = value

os.environ.setdefault('SWANLAB_PROJECT', 'gsm8k-grpo')

if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)

print('SWANLAB_PROJECT =', os.environ.get('SWANLAB_PROJECT'))
print('HF_TOKEN set =', bool(os.environ.get('HF_TOKEN')))
print('SWANLAB_API_KEY set =', bool(os.environ.get('SWANLAB_API_KEY')))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


SWANLAB_PROJECT = gsm8k-grpo
HF_TOKEN set = True
SWANLAB_API_KEY set = True


## Install dependencies

In [5]:
!pip uninstall -y torchao
!pip install -q -U "trl[vllm]>=0.29.0" "accelerate>=1.4.0" swanlab bitsandbytes sentencepiece
!pip install -q -e RL_common

import torch
import trl
import vllm
import accelerate

print('torch =', torch.__version__)
print('trl =', trl.__version__)
print('vllm import ok')
print('cuda available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu =', torch.cuda.get_device_name(0))

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 371.2/371.2 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.1/161.1 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.2/433.2 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.1 MB/s eta 0:00:00
   

## Validate DAPO config

In [6]:
import inspect
import yaml
from pathlib import Path
from trl import GRPOConfig
from trl.rewards import get_soft_overlong_punishment

BASE_CONFIG = PROJECT_DIR / 'RL_GRPO_train/configs/qwen25_3b_sft_grpo.yaml'
ACTIVE_CONFIG = PROJECT_DIR / 'RL_GRPO_train/configs/qwen25_3b_sft_dapo.yaml'

ALLOWED_DAPO_DIFFS = {
    ('run', 'name'),
    ('grpo', 'run_name'),
    ('grpo', 'loss_type'),
    ('grpo', 'mask_truncated_completions'),
    ('grpo', 'epsilon'),
    ('grpo', 'epsilon_high'),
    ('reward', 'soft_overlong_punishment'),
}

def changed_paths(left, right, prefix=()):
    if isinstance(left, dict) and isinstance(right, dict):
        keys = set(left) | set(right)
        for key in sorted(keys):
            yield from changed_paths(left.get(key), right.get(key), prefix + (key,))
    elif left != right:
        yield prefix

base_cfg = yaml.safe_load(BASE_CONFIG.read_text(encoding='utf-8'))
cfg = yaml.safe_load(ACTIVE_CONFIG.read_text(encoding='utf-8'))

unexpected = sorted(set(changed_paths(base_cfg, cfg)) - ALLOWED_DAPO_DIFFS)
if unexpected:
    raise AssertionError(f'Unexpected non-DAPO config changes: {unexpected}')

signature = inspect.signature(GRPOConfig.__init__)
accepts_kwargs = any(param.kind == inspect.Parameter.VAR_KEYWORD for param in signature.parameters.values())
required_args = {'loss_type', 'mask_truncated_completions', 'epsilon', 'epsilon_high'}
missing_args = set() if accepts_kwargs else required_args - set(signature.parameters)
if missing_args:
    raise RuntimeError(f'The installed TRL GRPOConfig does not support DAPO fields: {sorted(missing_args)}')

soft_cfg = cfg['reward']['soft_overlong_punishment']
if not soft_cfg.get('enabled', True):
    raise AssertionError('DAPO soft_overlong_punishment must be enabled')
if int(soft_cfg['max_completion_len']) != int(cfg['grpo']['max_completion_length']):
    raise AssertionError('soft_overlong max_completion_len must match grpo.max_completion_length')

print('ACTIVE_CONFIG =', ACTIVE_CONFIG)
print(ACTIVE_CONFIG.read_text(encoding='utf-8'))

ACTIVE_CONFIG = /content/drive/MyDrive/HPML_project/RL_GRPO_train/configs/qwen25_3b_sft_dapo.yaml
run:
  name: qwen25_3b_instruct_sft_dapo_g{grpo.num_generations}_train{train_dataset.limit}
  output_root: RL_GRPO_train/outputs

model:
  base_model_name_or_path: Qwen/Qwen2.5-3B-Instruct
  adapter_path: SFT_train/outputs/qwen25_3b_gsm8k_lora_sft_full
  tokenizer_name_or_path: Qwen/Qwen2.5-3B-Instruct
  trust_remote_code: true
  prompt_template: qwen_chat
  system_prompt: ""
  include_empty_system: false
  dtype: bf16
  device_map:
  load_in_4bit: false
  is_trainable_adapter: true

train_dataset:
  kind: sft_train
  name: openai/gsm8k
  config: main
  path: SFT_train/data/gsm8k_sft_train.json
  limit:

eval_dataset:
  kind: sft_val
  name: openai/gsm8k
  config: main
  path: SFT_train/data/gsm8k_sft_val.json
  limit: 725

final_eval_dataset:
  kind: gsm8k_test
  name: openai/gsm8k
  config: main
  start_index: 0
  limit:

reward:
  answer_correct: 1.0
  answer_incorrect: 0.0
  format_rew

## Train

In [7]:
!accelerate launch --num_processes 1 RL_GRPO_train/train_grpo.py --config "{ACTIVE_CONFIG}"

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-05-05 02:08:55.590344: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-05 02:08:55.664304: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
tokenizer_config.json: 7.30kB [00:00

## Inspect outputs

In [8]:
import json
import sys
from pathlib import Path

COMMON_SRC = PROJECT_DIR / 'RL_common/src'
if str(COMMON_SRC) not in sys.path:
    sys.path.insert(0, str(COMMON_SRC))

from rl_common.config import format_run_name, load_config, make_run_dir

cfg = load_config(str(ACTIVE_CONFIG))
cfg['run']['resolved_name'] = format_run_name(cfg['run']['name'], cfg)
run_dir = make_run_dir(cfg['run']['output_root'], cfg['run']['resolved_name'])
print('run_dir =', run_dir)
!find "{run_dir}" -maxdepth 2 -type f | sort

summary_path = run_dir / 'train_summary.json'
if summary_path.exists():
    print(summary_path.read_text(encoding='utf-8'))

best_path = run_dir / 'best_eval_results.json'
if best_path.exists():
    data = json.loads(best_path.read_text(encoding='utf-8'))
    print(json.dumps(data['summary'], indent=2, ensure_ascii=False))

run_dir = /content/drive/MyDrive/HPML_project/RL_GRPO_train/outputs/qwen25_3b_instruct_sft_dapo_g16_trainall
/content/drive/MyDrive/HPML_project/RL_GRPO_train/outputs/qwen25_3b_instruct_sft_dapo_g16_trainall/best_eval_results.json
/content/drive/MyDrive/HPML_project/RL_GRPO_train/outputs/qwen25_3b_instruct_sft_dapo_g16_trainall/final_adapter/adapter_config.json
/content/drive/MyDrive/HPML_project/RL_GRPO_train/outputs/qwen25_3b_instruct_sft_dapo_g16_trainall/final_adapter/adapter_model.safetensors
/content/drive/MyDrive/HPML_project/RL_GRPO_train/outputs/qwen25_3b_instruct_sft_dapo_g16_trainall/final_adapter/README.md
/content/drive/MyDrive/HPML_project/RL_GRPO_train/outputs/qwen25_3b_instruct_sft_dapo_g16_trainall/latest_eval_results.json
/content/drive/MyDrive/HPML_project/RL_GRPO_train/outputs/qwen25_3b_instruct_sft_dapo_g16_trainall/resolved_config.yaml
/content/drive/MyDrive/HPML_project/RL_GRPO_train/outputs/qwen25_3b_instruct_sft_dapo_g16_trainall/train_summary.json
{
  "global_

## Optional final eval on official GSM8K test

In [9]:
RUN_FINAL_TEST = True

if RUN_FINAL_TEST:
    cfg = load_config(str(ACTIVE_CONFIG))
    cfg['run']['resolved_name'] = format_run_name(cfg['run']['name'], cfg)
    train_run_dir = make_run_dir(cfg['run']['output_root'], cfg['run']['resolved_name'])
    final_adapter = train_run_dir / 'final_adapter'
    if not final_adapter.exists():
        raise FileNotFoundError(f'Missing final adapter: {final_adapter}')

    cfg['model']['adapter_path'] = str(final_adapter)
    cfg['model']['tokenizer_name_or_path'] = cfg['model']['base_model_name_or_path']
    cfg['run']['name'] = cfg['run']['resolved_name'] + '_final_test'
    cfg['eval']['max_new_tokens'] = 256
    cfg['eval']['batch_size'] = 512

    eval_cfg = Path('/content/dapo_final_eval.yaml')
    eval_cfg.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
    print(eval_cfg.read_text(encoding='utf-8')[:1500])
    !python RL_GRPO_train/train_grpo.py --config "{eval_cfg}" --eval-only --final-test

run:
  name: qwen25_3b_instruct_sft_dapo_g16_trainall_final_test
  output_root: RL_GRPO_train/outputs
  resolved_name: qwen25_3b_instruct_sft_dapo_g16_trainall
model:
  base_model_name_or_path: Qwen/Qwen2.5-3B-Instruct
  adapter_path: /content/drive/MyDrive/HPML_project/RL_GRPO_train/outputs/qwen25_3b_instruct_sft_dapo_g16_trainall/final_adapter
  tokenizer_name_or_path: Qwen/Qwen2.5-3B-Instruct
  trust_remote_code: true
  prompt_template: qwen_chat
  system_prompt: ''
  include_empty_system: false
  dtype: bf16
  device_map: null
  load_in_4bit: false
  is_trainable_adapter: true
train_dataset:
  kind: sft_train
  name: openai/gsm8k
  config: main
  path: SFT_train/data/gsm8k_sft_train.json
  limit: null
eval_dataset:
  kind: sft_val
  name: openai/gsm8k
  config: main
  path: SFT_train/data/gsm8k_sft_val.json
  limit: 725
final_eval_dataset:
  kind: gsm8k_test
  name: openai/gsm8k
  config: main
  start_index: 0
  limit: null
reward:
  answer_correct: 1.0
  answer_incorrect: 0.0
  fo